# ISAC-MIMO DDPG Training — Google Colab

**Pipeline:**
1. ✅ GPU check
2. 📦 Install dependencies
3. 📁 Mount Google Drive (for model persistence)
4. 🔧 Project setup
5. 🚀 Run DDPG training (`Nt=4`, `Nr=4`, `200 000` timesteps)
6. 📈 TensorBoard reward curve (inline)
7. 💾 Copy saved models to Google Drive

> **Assumption:** the project has been uploaded to `/content/isac-mimo-drl`.  
> If you cloned it from GitHub, replace the upload step with `!git clone <your-repo-url> /content/isac-mimo-drl`.

---
## Cell 1 — GPU Check

In [ ]:
import subprocess
import sys

# ── CUDA availability via nvidia-smi ──────────────────────────────────────────
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅  nvidia-smi output:")
    print(result.stdout)
else:
    print("⚠️  nvidia-smi not found — GPU may not be enabled.")
    print("    Go to Runtime → Change runtime type → Hardware accelerator → GPU")

# ── PyTorch CUDA check (run after install; skip gracefully if not yet present) ─
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    device   = torch.cuda.get_device_name(0) if cuda_ok else "CPU"
    status   = "✅" if cuda_ok else "⚠️ "
    print(f"{status} torch.cuda.is_available() = {cuda_ok}  |  device = {device}")
except ImportError:
    print("ℹ️  PyTorch not installed yet — will be confirmed after Cell 2.")

---
## Cell 2 — Install Dependencies

Colab ships with an older pip; we pin versions that are mutually compatible with CUDA 12.x.

In [ ]:
# Upgrade pip first to avoid resolver warnings
!pip install --quiet --upgrade pip

# Core scientific stack (Colab already has numpy/scipy but pin for safety)
!pip install --quiet \
    numpy \
    scipy \
    matplotlib \
    tqdm

# RL / environment stack
!pip install --quiet \
    gymnasium \
    "stable-baselines3[extra]" \
    tensorboard

# PyTorch with CUDA 12.1 (matches Colab T4/A100 images as of 2025)
!pip install --quiet \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

# Misc project dependencies
!pip install --quiet cvxpy python-dotenv

# ── Confirm CUDA is now visible to PyTorch ────────────────────────────────────
import torch
cuda_ok = torch.cuda.is_available()
device   = torch.cuda.get_device_name(0) if cuda_ok else "CPU"
print(f"\n{'✅' if cuda_ok else '⚠️ '} PyTorch {torch.__version__}  |  "
      f"CUDA available: {cuda_ok}  |  Device: {device}")

---
## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

import os

# Folder inside My Drive where models will be saved
GDRIVE_MODEL_DIR = "/content/drive/MyDrive/isac_mimo_drl/models"
os.makedirs(GDRIVE_MODEL_DIR, exist_ok=True)
print(f"✅  Google Drive mounted.  Models will be saved to:\n    {GDRIVE_MODEL_DIR}")

---
## Cell 4 — Project Setup

Add the project root to `sys.path` so all local imports resolve correctly.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/isac-mimo-drl")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} not found.\n"
        "Upload the project folder to /content/isac-mimo-drl, or run:\n"
        "  !git clone <your-repo-url> /content/isac-mimo-drl"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Verify key modules are importable
from environment.isac_env import ISACEnv          # noqa: F401
from environment.mimo_system import MIMOSystem     # noqa: F401
from training.train_ddpg import make_isac_env      # noqa: F401

print(f"✅  Project root on path: {PROJECT_ROOT}")
print("✅  All local imports resolved successfully.")

---
## Cell 5 — Run DDPG Training

**Config:** `Nt=4`, `Nr=4`, `total_timesteps=200 000`, action noise σ=0.1, eval every 10 000 steps.

TensorBoard logs are written to `/content/isac-mimo-drl/logs/ddpg_isac/`.

In [ ]:
import time

import numpy as np
import gymnasium as gym
from stable_baselines3 import DDPG
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from environment.channel_model import SVChannelModel
from environment.isac_env import ISACEnv
from environment.mimo_system import MIMOSystem
from environment.v2x_scenario import V2XScenario
from utils.reward_config import RewardConfig

# ── Paths ──────────────────────────────────────────────────────────────────────
LOG_DIR   = PROJECT_ROOT / "logs" / "ddpg_isac"
MODEL_DIR = PROJECT_ROOT / "models"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
Nt               = 4
Nr               = 4
TOTAL_TIMESTEPS  = 200_000
LEARNING_RATE    = 1e-3
BATCH_SIZE       = 256
BUFFER_SIZE      = 100_000
ACTION_NOISE_STD = 0.1
EVAL_FREQ        = 10_000
PRINT_FREQ       = 5_000

# ── Progress callback ─────────────────────────────────────────────────────────
class ProgressCallback(BaseCallback):
    """Prints a one-liner every PRINT_FREQ environment steps."""

    def __init__(self, print_freq: int = 5_000) -> None:
        super().__init__()
        self.print_freq = print_freq

    def _on_step(self) -> bool:
        if self.num_timesteps % self.print_freq == 0:
            ep_info    = self.locals.get("infos", [{}])[-1]
            ep_reward  = ep_info.get("episode", {}).get("r", float("nan"))
            print(f"[Step {self.num_timesteps:>7,}]  episode_reward = {ep_reward:>10.4f}")
        return True


# ── Environment factory ───────────────────────────────────────────────────────
def build_env() -> gym.Env:
    """Instantiate ISACEnv with Nt=4, Nr=4 (calibration runs in __init__)."""
    mimo     = MIMOSystem(Nt=Nt, Nr=Nr, carrier_freq=28e9)
    scenario = V2XScenario(speed_kmh=60.0, update_interval=0.1)
    channel  = SVChannelModel(
        mimo=mimo, distance=100.0,
        num_clusters=3, rays_per_cluster=10, azimuth_spread_deg=10.0,
    )
    reward_cfg = RewardConfig(alpha=0.5, beta=0.5)
    return ISACEnv(mimo=mimo, scenario=scenario, channel=channel,
                   reward_config=reward_cfg, max_steps=200)


print("Building environment (calibration loop runs inside ISACEnv.__init__) …")
env = build_env()
env = Monitor(env)
env = DummyVecEnv([lambda: env])
env = VecNormalize(env, norm_obs=True, norm_reward=False)
print(f"✅  Env ready  |  obs shape: {env.observation_space.shape}  "
      f"|  action shape: {env.action_space.shape}")

# ── Action noise ──────────────────────────────────────────────────────────────
n_actions    = env.action_space.shape[-1]
action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=ACTION_NOISE_STD * np.ones(n_actions),
)

# ── DDPG model ────────────────────────────────────────────────────────────────
model = DDPG(
    "MlpPolicy",
    env,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    buffer_size=BUFFER_SIZE,
    action_noise=action_noise,
    tensorboard_log=str(LOG_DIR),
    verbose=0,   # silenced; ProgressCallback handles output
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print(f"✅  DDPG agent built  |  device: {model.device}")

# ── Callbacks ─────────────────────────────────────────────────────────────────
progress_cb = ProgressCallback(print_freq=PRINT_FREQ)
eval_cb     = EvalCallback(
    env,
    best_model_save_path=str(MODEL_DIR / "ddpg_isac_best"),
    log_path=str(LOG_DIR),
    eval_freq=EVAL_FREQ,
    deterministic=True,
    render=False,
)

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\n🚀  Starting DDPG training for {TOTAL_TIMESTEPS:,} timesteps …\n" + "─" * 55)
t0 = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[progress_cb, eval_cb],
    reset_num_timesteps=True,
)
elapsed = time.time() - t0
print("─" * 55)
print(f"✅  Training complete in {elapsed:.1f} s  ({elapsed/60:.1f} min)")

# ── Save final model + VecNormalize stats ─────────────────────────────────────
FINAL_DIR = MODEL_DIR / "ddpg_isac_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
model.save(str(FINAL_DIR / "model"))
env.save(str(FINAL_DIR / "vecnormalize.pkl"))
print(f"💾  Final model  → {FINAL_DIR / 'model.zip'}")
print(f"💾  VecNormalize → {FINAL_DIR / 'vecnormalize.pkl'}")

env.close()

---
## Cell 6 — TensorBoard Reward Curve (Inline)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/isac-mimo-drl/logs/ddpg_isac --port 6006

---
## Cell 7 — Plot Episode Reward Curve from TensorBoard Logs

Reads the `eval/mean_reward` scalar directly from the saved TensorBoard event files and renders a matplotlib figure — useful if TensorBoard inline doesn't load.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt
import glob, os

# Locate the most recent event file written by EvalCallback
event_files = sorted(
    glob.glob(str(LOG_DIR / "**" / "events.out.tfevents.*"), recursive=True)
)

if not event_files:
    print("⚠️  No TensorBoard event files found yet — run Cell 5 first.")
else:
    ea = EventAccumulator(str(LOG_DIR))
    ea.Reload()

    available_tags = ea.Tags().get("scalars", [])
    print(f"Available scalar tags: {available_tags}")

    # Try eval/mean_reward first; fall back to rollout/ep_rew_mean
    tag = next(
        (t for t in ["eval/mean_reward", "rollout/ep_rew_mean"] if t in available_tags),
        available_tags[0] if available_tags else None,
    )

    if tag is None:
        print("⚠️  No recognisable reward scalar found in event files.")
    else:
        events = ea.Scalars(tag)
        steps  = [e.step  for e in events]
        values = [e.value for e in events]

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(steps, values, linewidth=1.8, color="#4C8EDA", label=tag)
        ax.set_xlabel("Environment Steps", fontsize=12)
        ax.set_ylabel("Mean Episode Reward", fontsize=12)
        ax.set_title("DDPG Training — ISAC-MIMO (Nt=4, Nr=4)", fontsize=14)
        ax.legend(fontsize=11)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig("/content/isac-mimo-drl/logs/reward_curve.png", dpi=150)
        plt.show()
        print(f"✅  Reward curve saved to /content/isac-mimo-drl/logs/reward_curve.png")

---
## Cell 8 — Copy Models to Google Drive

In [ ]:
import shutil, os

LOCAL_MODEL_DIR = str(MODEL_DIR)

def copy_to_drive(src_name: str) -> None:
    """Recursively copy a model folder from local to Google Drive."""
    src  = os.path.join(LOCAL_MODEL_DIR, src_name)
    dst  = os.path.join(GDRIVE_MODEL_DIR, src_name)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)           # overwrite cleanly
        shutil.copytree(src, dst)
        print(f"✅  {src_name:30s} → {dst}")
    else:
        print(f"⚠️  {src_name:30s} not found at {src} — was training completed?")

print("Copying models to Google Drive …")
copy_to_drive("ddpg_isac_best")
copy_to_drive("ddpg_isac_final")

# Also copy the reward curve plot if it exists
plot_src = "/content/isac-mimo-drl/logs/reward_curve.png"
plot_dst = os.path.join(GDRIVE_MODEL_DIR, "reward_curve.png")
if os.path.exists(plot_src):
    shutil.copy2(plot_src, plot_dst)
    print(f"✅  {'reward_curve.png':30s} → {plot_dst}")

print("\n🎉  All artefacts persisted to Google Drive.")

---
## Cell 9 — Summary Statistics

In [ ]:
import numpy as np

# Reload eval rewards logged by EvalCallback (stored in evaluations.npz)
eval_npz_path = LOG_DIR / "evaluations.npz"

if eval_npz_path.exists():
    data    = np.load(str(eval_npz_path))
    results = data["results"]        # shape: (n_evals, n_eval_episodes)
    steps   = data["timesteps"]      # shape: (n_evals,)

    mean_per_eval = results.mean(axis=1)
    best_idx      = mean_per_eval.argmax()

    print("=" * 50)
    print("  DDPG Training — Summary Statistics")
    print("=" * 50)
    print(f"  Total timesteps trained : {TOTAL_TIMESTEPS:>10,}")
    print(f"  Eval checkpoints        : {len(steps):>10}")
    print(f"  Final eval mean reward  : {mean_per_eval[-1]:>10.4f}")
    print(f"  Best  eval mean reward  : {mean_per_eval[best_idx]:>10.4f}  "
          f"(at step {steps[best_idx]:,})")
    print(f"  Reward std (final eval) : {results[-1].std():>10.4f}")
    print(f"  Min eval mean reward    : {mean_per_eval.min():>10.4f}")
    print("=" * 50)
else:
    print(f"⚠️  evaluations.npz not found at {eval_npz_path}")
    print("    Run Cell 5 (training) first, or check LOG_DIR path.")